# 00 - Exploration (Qwen3.5-4B-Base + Vietnamese dataset)

Notebook khám phá dataset, tokenizer, kiến trúc model và LoRA param count — chạy TRƯỚC khi training.
Bám sát English Llama 3.2 fine-tune reference (`Fine_tune_Llama3_2_qlora_colab_fullcode.ipynb`).

**Target environment:** Colab T4 16GB. Khi load 32-bit base (~16GB), Accelerate tự offload phần thừa
qua CPU RAM (đủ trên Colab high-RAM), không OOM. Khi load 4-bit chỉ ~2.5GB.

**Sections:**
- A: Dataset preview (train/val/test counts, PROMPT/COMPLETION samples)
- B: Tokenizer investigation (digit-by-digit verification)
- C: QLoRA architecture (memory footprint 32-bit vs 4-bit, `print(base_model)`)
- D: Zero-shot single-sample predict (base model chưa fine-tune)
- E: LoRA parameter count (`print_trainable_parameters`)

In [ ]:
# Cell 1 - Imports + constants
import os, sys, torch
import numpy as np

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_4"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Section A — Dataset Exploration

In [ ]:
# Cell 2 - Load dataset splits
ds = load_dataset(DATASET_NAME)
train, val, test = ds["train"], ds["validation"], ds["test"]
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")
print(f"Columns: {train.column_names}")

In [ ]:
# Cell 3 - Display train[0] dict (English Cell 63)
train[0]

In [ ]:
# Cell 4 - Display test[0] dict (English Cell 61)
test[0]

In [ ]:
# Cell 5 - Display train[0].prompt only (English Cell 12)
print(train[0]["prompt"])

In [ ]:
# Cell 6 - Display train[1].completion (English Cell 13)
print(train[1]["completion"])

In [ ]:
# Cell 7 - PROMPT/COMPLETION sample (English Cell 14)
print("PROMPT:")
print(test[1]["prompt"])
print()
print("COMPLETION:")
print(test[1]["completion"])

## Section B — Tokenizer Investigation

English Llama 3.2 tokenize prices 0-999 thành **1 token mỗi số**. Qwen 3.5 thì sao?
Cell này xác nhận empirically — kết quả ảnh hưởng quyết định `MAX_NEW_TOKENS`.

In [ ]:
# Cell 8 - Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"EOS: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

In [ ]:
# Cell 9 - investigate_tokenizer (English Cell 26-30)
def investigate_tokenizer(model_name, tok):
    print(f"=== Investigating {model_name} ===")
    for n in [0, 1, 10, 100, 999, 1000]:
        ids = tok.encode(str(n), add_special_tokens=False)
        decoded = [tok.decode([i]) for i in ids]
        print(f"  '{n}'  -> {len(ids)} tokens  ids={ids}  decoded={decoded}")

investigate_tokenizer(BASE_MODEL, tokenizer)

In [ ]:
# Cell 10 - Token count distribution (English Cell 15)
counts = [
    len(tokenizer.encode(train[i]["prompt"] + train[i]["completion"]))
    for i in range(2000)
]
print(f"Token counts on 2000 train samples:")
print(f"  min={min(counts)}  max={max(counts)}  mean={np.mean(counts):.0f}  median={int(np.median(counts))}")
print(f"  p95={int(np.percentile(counts, 95))}  p99={int(np.percentile(counts, 99))}")
print(f"\nMAX_SEQ_LENGTH=192 (configured) covers {sum(c <= 192 for c in counts) / len(counts) * 100:.1f}% of samples")

## Section C — QLoRA Architecture & Memory Footprint

English Llama 3.2 3B fp32 = 12.9 GB → 4-bit NF4 = 2.20 GB.
Qwen 3.5 4B ở fp32 ~16 GB — trên Colab T4 16GB Accelerate sẽ tự offload phần thừa
qua CPU RAM (cần Colab High-RAM).

In [ ]:
# Cell 11 - Load 32-bit base model (English Cell 41-42)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Cell 12 - Display 32-bit base_model architecture (English Cell 43)
# Expected: Qwen3 stack with Linear layers (float32)
print(base_model)

In [ ]:
# Cell 13 - Del 32-bit + load 4-bit NF4 (English Cell 45-46)
del base_model
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Cell 14 - Display 4-bit base_model architecture (English Cell 47)
# Expected: same stack but Linear4bit replacing Linear
print(base_model)

## Section D — Zero-shot Single-Sample Predict (chưa fine-tune)

Base model chưa học format giá Việt — output có thể leak text thừa
(English: `'99.00 USD. The pedal is'`). Đây là baseline qualitative.

In [ ]:
# Cell 15 - Define model_predict (English Cell 66 / 105 pattern)
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
# Cell 16 - Display test[0] (English Cell 67)
test[0]

In [ ]:
# Cell 17 - Zero-shot predict on test[0] (English Cell 68)
model_predict(test[0])

## Section E — LoRA Parameters

Wrap 4-bit base với LoRA config (giống config trong `training_utils.py`) → kiểm tra số trainable params.

In [ ]:
# Cell 18 - Apply LoRA + print_trainable_parameters (English Cell 48-49)
LORA_R         = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0.1
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
peft_model = get_peft_model(base_model, lora_parameters)
peft_model.print_trainable_parameters()

In [ ]:
# Cell 19 - Display PEFT-wrapped model (English Cell 104 pattern, pre-training version)
# Expected: PeftModelForCausalLM wrapping LoraModel; lora_A / lora_B Linear layers in each target module.
print(peft_model)

## Summary

Nếu chạy hết notebook không lỗi:
- Dataset format & PROMPT/COMPLETION schema xác nhận
- Tokenizer Qwen 3.5 vs Llama 3.2 cho prices: xác nhận `MAX_NEW_TOKENS=8` đủ
- Memory footprint: ~16 GB (fp32) → ~2.5 GB (4-bit NF4) — đúng kỳ vọng
- LoRA trainable params: ~18M / 4B ≈ 0.45%
- Zero-shot base output có format chưa chuẩn (cần fine-tune)

**Next:** chạy `01_zero_shot.ipynb` để có MAE baseline trên 200 test items.